# PDF → Markdown + Images + LaTeX — v5 (cepat, hemat, akurat)

Marker tetap dijalankan via **CLI di subprocess** (kebal modul stale, seperti v4). Yang berubah:

| Masalah di v4 | Perbaikan v5 |
|---|---|
| `--force_ocr` selalu aktif: 85 halaman = 1135 s (0,07 hal/detik) | **Mode per PDF.** PDF digital memakai text layer (tanpa OCR ulang); scan dan PDF dengan teks rusak memakai OCR |
| T4 x2 aktif, hanya 1 GPU dipakai | **Semua GPU dipakai.** PDF dibagi seimbang menurut jumlah halaman, satu proses marker per GPU |
| Semua file diproses ulang tiap run | **Cache** (hash isi + mode + versi): file yang sudah jadi dilewati |
| GPU wajib (`assert`) | **`DEVICE`** = `auto` / `cuda` / `cpu` |
| Waktu per file = total/jumlah file | Estimasi waktu proporsional jumlah halaman |
| Log progress membanjiri notebook | Log ke `_logs/`, notebook hanya menampilkan ringkasan |
| `pip install` tiap run (~50 s) | pip hanya jika versi di disk berbeda |

**Parameter utama** (cell 2): `DEVICE`, `MODE`, `KEEP_OCR_FOR_MATH`, `WORKERS_PER_GPU`, `CLEAN_OUTPUT`.

- `MODE="auto"`: text layer ≥ 90% halaman, tidak rusak, rumus tidak padat → `fast`; selain itu → `accurate`.
- `KEEP_OCR_FOR_MATH=True` (bawaan): PDF dengan simbol matematika padat tetap `accurate` demi rumus inline. Paper berisi rumus (LBP, fuzzy) hampir pasti masuk jalur ini, jadi percepatan utamanya datang dari 2 GPU dan cache. Jalankan **cell kalibrasi** pada satu PDF untuk melihat apakah `fast` cukup; bila ya, set `False`.

**Pakai**
- Kaggle: attach dataset PDF, accelerator **GPU T4 x2**, Run All.
- Colab: runtime GPU T4, unggah PDF ke `/content/input`, Run All. Tanpa GPU: `DEVICE="cpu"`, `MODE="fast"`.

Kecepatan v5 belum diukur pada GPU nyata. Baseline v4 di atas berasal dari log run sebelumnya; bandingkan dengan baris `hal/detik` di akhir cell 2.

In [ ]:
# Install pinned. Hanya memanggil pip bila versi di disk berbeda -> hemat ~1 menit per run.
# Pipeline memakai subprocess, jadi versi di disk selalu yang dipakai (bukan modul di kernel).
import importlib.metadata as im, subprocess, sys

PINS = {"marker-pdf": "1.10.2", "surya-ocr": "0.17.1"}

def _ver(name):
    try:
        return im.version(name)
    except im.PackageNotFoundError:
        return None

need = [f"{k}=={v}" for k, v in PINS.items() if _ver(k) != v]
if _ver("pymupdf4llm") is None:
    need.append("pymupdf4llm")
if need:
    print("pip install", need)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
else:
    print("Dependensi sudah sesuai, pip dilewati.")

for n in ("marker-pdf", "surya-ocr", "pymupdf4llm"):
    print(f"{n:12s}: {_ver(n)}")
try:
    print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip())
except FileNotFoundError:
    print("nvidia-smi tidak ada: runtime tanpa GPU (DEVICE akan jatuh ke CPU)")

In [ ]:
"""
PDF -> Markdown + images/ + LaTeX   (v5: cepat, hemat, akurat)
Marker dijalankan via CLI di SUBPROCESS (kebal modul stale di kernel).
Output: <WORK>/output/<slug>/{<slug>.md, images/, <slug>_meta.json}

Optimasi dibanding v4:
  1. Mode per-PDF: PDF digital memakai text layer (tanpa OCR ulang), PDF scan memakai OCR.
  2. Semua GPU dipakai: PDF dibagi seimbang (per jumlah halaman), satu proses marker per GPU.
  3. Cache: PDF yang sudah diproses (isi + mode + versi sama) dilewati.
  4. DEVICE bisa "auto" | "cuda" | "cpu".
  5. Waktu per file dihitung dari proporsi halaman, bukan total/len yang sama untuk semua.
"""

import hashlib, importlib.metadata as im, json, os, re, shutil, subprocess, sys, time, unicodedata
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# ================================================================== CONFIG
DEVICE = "auto"            # "auto" | "cuda" | "cpu"
MODE   = "auto"            # "auto" | "fast" | "accurate"
#   fast     : pakai text layer PDF, OCR hanya bila marker menilai perlu (jauh lebih cepat)
#   accurate : --force_ocr di semua halaman (lambat; terbaik untuk scan & inline math)
#   auto     : putuskan per PDF dari analisis text layer (lihat ambang di bawah)

KEEP_OCR_FOR_MATH = True   # auto: PDF dengan simbol matematika padat tetap "accurate"
                           # Set False untuk kecepatan maksimum; cek dulu dengan cell kalibrasi.
MIN_TEXT_COVERAGE = 0.90   # auto: min. porsi halaman yang punya text layer
MIN_CHARS_PER_PAGE = 100   # halaman dianggap punya teks bila >= jumlah karakter ini
MAX_GARBLE_RATIO  = 0.02   # auto: maks. rasio karakter rusak (U+FFFD, (cid:..))
MATH_THRESHOLD    = 0.004  # auto: rasio karakter matematika/Yunani/private-use -> "rumus padat"
ANALYZE_SAMPLE    = 30     # jumlah halaman maksimum yang dianalisis per PDF
ACCURATE_COST     = 3.0    # perkiraan biaya relatif halaman accurate vs fast (untuk membagi beban GPU)

WORKERS_PER_GPU = 1        # naikkan ke 2 hanya jika VRAM cukup (tiap worker memuat model sendiri)
CPU_WORKERS     = 2        # jumlah worker bila DEVICE cpu
DISABLE_IMAGES  = False    # True = jangan ekstrak gambar (sedikit lebih cepat)
CLEAN_OUTPUT    = False    # True = hapus output & cache lama sebelum run
BATCH_ENV = {}             # opsional, ukuran batch surya, mis. {"RECOGNITION_BATCH_SIZE": "256"}
                           # nilai bawaan sudah disetel; naikkan hati-hati (risiko OOM di T4)

INPUT_DIR = None           # None = deteksi otomatis (Kaggle: /kaggle/input, Colab: /content/input)
MARKER_VERSION = "1.10.2"
# ====================================================================================

if INPUT_DIR is None:
    if Path("/kaggle").exists():
        INPUT_DIR, WORK = Path("/kaggle/input"), Path("/kaggle/working")
    elif Path("/content").exists():
        INPUT_DIR, WORK = Path("/content/input"), Path("/content/work")
    else:
        INPUT_DIR, WORK = Path("input"), Path("work")
else:
    INPUT_DIR = Path(INPUT_DIR)
    WORK = Path("/kaggle/working") if Path("/kaggle").exists() else Path("/content/work") \
        if Path("/content").exists() else Path("work")

OUTPUT_DIR = WORK / "output"
STAGING    = WORK / "_staging"
MARKER_RAW = WORK / "_marker_raw"
LOG_DIR    = WORK / "_logs"

try:
    import pymupdf
except ImportError:                      # pymupdf < 1.24
    import fitz as pymupdf

MATH_CHARS = set("∑∏∫∂∇√∞≈≠≤≥±×÷∈∉⊂⊃⊆⊇∪∩∧∨¬∀∃→←↔⇒⇔≡∝∗⊗⊕"
                 "αβγδεζηθικλμνξοπρστυφχψωΑΒΓΔΘΛΞΠΣΦΨΩµ")


# ------------------------------------------------------------------ util
def slugify(stem: str) -> str:
    s = unicodedata.normalize("NFKD", stem).encode("ascii", "ignore").decode()
    s = re.sub(r"[^A-Za-z0-9._-]+", "_", s).strip("_")
    return s[:80] or "doc"


def sha1_file(p: Path) -> str:
    h = hashlib.sha1()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def qc_report(md: str, n_images: int) -> dict:
    headings = re.findall(r"^#{1,6}\s+(.+)$", md, flags=re.M)
    return {
        "chars": len(md),
        "headings": len(headings), "heading_titles": headings[:30],
        "display_equations": len(re.findall(r"\$\$.+?\$\$", md, flags=re.S)),
        "inline_equations": len(re.findall(r"(?<!\$)\$(?!\$)[^$\n]+?\$(?!\$)", md)),
        "markdown_table_rows": len(re.findall(r"^\|.+\|\s*$", md, flags=re.M)),
        "html_tables": md.lower().count("<table"),
        "images_saved": n_images,
        "references_section_found": bool(re.search(
            r"^#{1,6}\s*\**\s*(references|daftar pustaka|bibliography|literature cited)",
            md, flags=re.M | re.I)),
    }


# ------------------------------------------------------------------ device
def detect_gpus():
    """GPU CUDA dengan compute capability >= 7.0 (T4 ke atas). P100 (6.0) dilewati."""
    for query, has_cap in (("index,name,compute_cap", True), ("index,name", False)):
        try:
            r = subprocess.run(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader"],
                               capture_output=True, text=True, timeout=30)
        except (FileNotFoundError, subprocess.TimeoutExpired):
            return []
        if r.returncode == 0:
            break
    else:
        return []
    gpus = []
    for line in r.stdout.strip().splitlines():
        parts = [x.strip() for x in line.split(",")]
        idx, name = int(parts[0]), parts[1]
        if has_cap:
            try:
                if float(parts[2]) < 7.0:
                    print(f"[skip] GPU {idx} {name}: compute capability {parts[2]} < 7.0")
                    continue
            except ValueError:
                pass
        gpus.append((idx, name))
    return gpus


def plan_devices():
    gpus = detect_gpus() if DEVICE in ("auto", "cuda") else []
    if DEVICE == "cuda" and not gpus:
        raise SystemExit("DEVICE='cuda' tetapi tidak ada GPU yang didukung. "
                         "Aktifkan accelerator GPU T4 atau set DEVICE='cpu'.")
    if gpus:
        return [{"kind": "cuda", "idx": i, "name": n} for i, n in gpus]
    if DEVICE == "auto":
        print("PERINGATAN: GPU tidak ditemukan, memakai CPU (lambat untuk mode accurate).")
    return [{"kind": "cpu", "idx": None, "name": "CPU"}]


# ------------------------------------------------------------------ analisis PDF
def analyze(pdf: Path):
    try:
        doc = pymupdf.open(pdf)
        if doc.needs_pass:
            doc.close(); return None
        n = doc.page_count
        if n == 0:
            doc.close(); return None
        idxs = list(range(0, n, max(1, n // ANALYZE_SAMPLE)))
        chars = garbled = mathc = good = 0
        for i in idxs:
            t = doc[i].get_text("text")
            L = len(t.strip())
            chars += L
            good += L >= MIN_CHARS_PER_PAGE
            garbled += t.count("�") + 6 * t.count("(cid:")
            mathc += sum(1 for c in t if c in MATH_CHARS or "" <= c <= "")
        doc.close()
        return {"pages": n, "text_cov": good / len(idxs),
                "garble": garbled / max(chars, 1), "math": mathc / max(chars, 1)}
    except Exception as e:                                   # PDF korup, dll.
        print(f"[analisis gagal] {pdf.name}: {type(e).__name__}: {e}")
        return None


def decide_mode(a):
    if MODE in ("fast", "accurate"):
        return MODE, "manual"
    if a is None:
        return "accurate", "analisis gagal -> aman"
    if a["text_cov"] < MIN_TEXT_COVERAGE:
        return "accurate", f"text layer {a['text_cov']:.0%} (scan?)"
    if a["garble"] > MAX_GARBLE_RATIO:
        return "accurate", f"teks rusak {a['garble']:.1%}"
    if KEEP_OCR_FOR_MATH and a["math"] > MATH_THRESHOLD:
        return "accurate", f"rumus padat {a['math']:.2%}"
    return "fast", f"text layer sehat ({a['text_cov']:.0%}), rumus {a['math']:.2%}"


# ------------------------------------------------------------------ marker CLI
def build_cmd(mode, staging, raw, workers, extra=()):
    cmd = ["marker", str(staging), "--output_dir", str(raw),
           "--output_format", "markdown", "--workers", str(workers)]
    if mode == "accurate":
        cmd.append("--force_ocr")
    if DISABLE_IMAGES:
        cmd.append("--disable_image_extraction")
    cmd.extend(extra)
    return cmd


def build_env(device):
    env = dict(os.environ, **{k: str(v) for k, v in BATCH_ENV.items()})
    if device["kind"] == "cuda":
        env["CUDA_VISIBLE_DEVICES"] = str(device["idx"])     # tiap proses melihat 1 GPU
        env["TORCH_DEVICE"] = "cuda"
    else:
        env["CUDA_VISIBLE_DEVICES"] = ""
        env["TORCH_DEVICE"] = "cpu"
    return env


def run_batch(b):
    """Jalankan satu batch marker; log ke file, bukan ke notebook."""
    t0 = time.time()
    log = LOG_DIR / f"{b['name']}.log"
    print(f"[{b['name']}] mulai: {len(b['jobs'])} PDF, {b['pages']} hal, mode={b['mode']}")
    try:
        with open(log, "w", encoding="utf-8") as f:
            f.write("$ " + " ".join(b["cmd"]) + "\n")
            f.flush()
            p = subprocess.run(b["cmd"], env=b["env"], stdout=f, stderr=subprocess.STDOUT, text=True)
        rc = p.returncode
    except FileNotFoundError:
        rc = 127
        log.write_text("perintah 'marker' tidak ditemukan. Jalankan ulang cell install.\n")
    b["rc"], b["elapsed"] = rc, time.time() - t0
    print(f"[{b['name']}] selesai exit={rc} {b['elapsed']:.0f}s  (log: {log.name})")
    if rc != 0:
        print("".join(log.read_text(errors="ignore").splitlines(True)[-25:]))
    return b


def run_device(batches):                 # batch pada GPU yang sama berjalan berurutan
    return [run_batch(b) for b in batches]


# ------------------------------------------------------------------ pasca-proses
IMG_EXT = {".png", ".jpg", ".jpeg", ".webp", ".gif"}


def collect_marker_result(slug, raw_dirs):
    for raw in raw_dirs:
        cand = raw / slug
        if cand.is_dir() and list(cand.glob("*.md")):
            return cand
    return None


def finalize_marker(job, raw_dir, est_sec):
    out_dir = job["out_dir"]
    md = next(raw_dir.glob("*.md")).read_text(encoding="utf-8")
    img_dir = out_dir / "images"
    img_dir.mkdir(parents=True, exist_ok=True)
    n = 0
    for f in raw_dir.iterdir():
        if f.suffix.lower() in IMG_EXT:
            shutil.copy2(f, img_dir / f.name)
            md = md.replace(f"]({f.name})", f"](images/{f.name})")
            n += 1
    (out_dir / f"{job['slug']}.md").write_text(md, encoding="utf-8")
    return qc_report(md, n), est_sec


FALLBACK_SCRIPT = r"""
import sys, pathlib, pymupdf4llm
staged, out_dir = sys.argv[1], pathlib.Path(sys.argv[2])
img_dir = out_dir / "images"; img_dir.mkdir(parents=True, exist_ok=True)
md = pymupdf4llm.to_markdown(staged, write_images=True, image_path=str(img_dir),
                             image_format="png", table_strategy="lines_strict")
md = md.replace(str(img_dir) + "/", "images/")
(out_dir / (pathlib.Path(staged).stem + ".md")).write_text(md, encoding="utf-8")
"""


def run_fallback(job):
    fb = WORK / "_fb.py"
    fb.write_text(FALLBACK_SCRIPT)
    staged = job["src"]
    t0 = time.time()
    r = subprocess.run([sys.executable, str(fb), str(staged), str(job["out_dir"])],
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.returncode != 0:
        print(r.stdout[-2000:])
        return None, 0.0
    # nama file keluaran fallback = stem PDF sumber -> samakan dengan slug
    produced = job["out_dir"] / f"{Path(staged).stem}.md"
    target = job["out_dir"] / f"{job['slug']}.md"
    if produced != target and produced.exists():
        produced.replace(target)
    md = target.read_text(encoding="utf-8")
    n = len(list((job["out_dir"] / "images").glob("*")))
    return qc_report(md, n), time.time() - t0


# ================================================================== MAIN
t_start = time.time()

try:
    _mv = im.version("marker-pdf")
except im.PackageNotFoundError:
    _mv = None
if _mv != MARKER_VERSION:
    raise SystemExit(f"marker-pdf terpasang: {_mv}, dibutuhkan {MARKER_VERSION}. Jalankan ulang cell install.")

if CLEAN_OUTPUT:
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
for d in (STAGING, MARKER_RAW, LOG_DIR):
    shutil.rmtree(d, ignore_errors=True)
for d in (OUTPUT_DIR, STAGING, MARKER_RAW, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

devices = plan_devices()
print("Device:", ", ".join(f"{d['kind']}:{d['idx']} {d['name']}" if d["idx"] is not None
                           else d["name"] for d in devices))

pdfs = sorted(p for p in INPUT_DIR.rglob("*.pdf") if p.stat().st_size > 0)
if not pdfs:
    raise SystemExit(f"Tidak ada PDF di {INPUT_DIR}")

# ---- 1) analisis + keputusan mode + cache --------------------------------------
jobs, skipped, used = [], [], set()
for p in pdfs:
    slug = slugify(p.stem)
    base, i = slug, 2
    while slug in used:
        slug = f"{base}_{i}"; i += 1
    used.add(slug)

    a = analyze(p)
    mode, reason = decide_mode(a)
    if DEVICE == "cpu" and mode == "accurate" and MODE == "auto":
        print(f"  catatan: {p.name} butuh OCR tetapi DEVICE=cpu -> akan lambat")
    pages = a["pages"] if a else 0
    sha = sha1_file(p)
    key = f"{sha}:{mode}:{MARKER_VERSION}:{int(DISABLE_IMAGES)}"
    out_dir = OUTPUT_DIR / slug
    meta_f = out_dir / f"{slug}_meta.json"
    if meta_f.exists() and (out_dir / f"{slug}.md").exists():
        try:
            if json.loads(meta_f.read_text()).get("cache_key") == key:
                skipped.append(slug)
                print(f"  cache : {p.name} (lewati)")
                continue
        except json.JSONDecodeError:
            pass
    shutil.rmtree(out_dir, ignore_errors=True)
    out_dir.mkdir(parents=True)
    jobs.append({"slug": slug, "src": p, "pages": pages, "mode": mode, "reason": reason,
                 "cache_key": key, "out_dir": out_dir, "analysis": a})
    print(f"  proses: {p.name}  {pages} hal -> {mode}  [{reason}]")

# ---- 2) bagi beban ke device (LPT: terbesar dulu ke device paling ringan) -------
load = [0.0] * len(devices)
assign = [[] for _ in devices]
cost = lambda j: max(j["pages"], 1) * (ACCURATE_COST if j["mode"] == "accurate" else 1.0)
for j in sorted(jobs, key=cost, reverse=True):
    k = load.index(min(load))
    assign[k].append(j)
    load[k] += cost(j)

per_device_batches = []
for k, dev in enumerate(devices):
    batches = []
    for mode in ("fast", "accurate"):
        js = [j for j in assign[k] if j["mode"] == mode]
        if not js:
            continue
        name = f"{dev['kind']}{dev['idx'] if dev['idx'] is not None else ''}_{mode}"
        stg, raw = STAGING / name, MARKER_RAW / name
        stg.mkdir(parents=True); raw.mkdir(parents=True)
        for j in js:
            shutil.copy2(j["src"], stg / f"{j['slug']}.pdf")
        workers = WORKERS_PER_GPU if dev["kind"] == "cuda" else CPU_WORKERS
        batches.append({"name": name, "jobs": js, "mode": mode, "raw": raw,
                        "pages": sum(j["pages"] for j in js),
                        "cmd": build_cmd(mode, stg, raw, workers), "env": build_env(dev)})
    if batches:
        per_device_batches.append(batches)

# ---- 3) jalankan semua device paralel ------------------------------------------
done_batches = []
if per_device_batches:
    with ThreadPoolExecutor(max_workers=len(per_device_batches)) as ex:
        for res in ex.map(run_device, per_device_batches):
            done_batches.extend(res)
raw_dirs = [b["raw"] for b in done_batches]
batch_of = {j["slug"]: b for b in done_batches for j in b["jobs"]}

# ---- 4) finalisasi + fallback ---------------------------------------------------
all_meta = []
for j in jobs:
    b = batch_of[j["slug"]]
    raw = collect_marker_result(j["slug"], raw_dirs)
    tot_pages = max(b["pages"], 1)
    est = b["elapsed"] * (max(j["pages"], 1) / tot_pages) if b["pages"] else b["elapsed"] / len(b["jobs"])
    if raw is not None:
        qc, est = finalize_marker(j, raw, est)
        engine = "marker"
    else:
        print(f"[{j['slug']}] hasil marker tidak ada -> fallback pymupdf4llm")
        qc, est = run_fallback(j)
        engine = "pymupdf4llm-fallback" if qc else "FAILED"
        if qc:
            qc["warning"] = "Equation TIDAK dalam LaTeX pada mode fallback."
        else:
            qc = {}
    meta = {"source_pdf": str(j["src"]), "engine": engine, "mode": j["mode"],
            "mode_reason": j["reason"], "pages": j["pages"], "device": b["name"],
            "elapsed_sec_est": round(est, 1),
            "cache_key": j["cache_key"] if engine == "marker" else None, **qc}
    if engine != "FAILED" and j["pages"] and qc.get("chars", 0) < 90 * j["pages"]:
        meta["warning_low_text"] = "Teks hasil sangat sedikit untuk jumlah halaman; cek manual / pakai mode accurate."
    (j["out_dir"] / f"{j['slug']}_meta.json").write_text(json.dumps(meta, indent=2))
    all_meta.append(meta)

# gabungkan dengan meta hasil cache agar summary.json lengkap
for slug in skipped:
    mf = OUTPUT_DIR / slug / f"{slug}_meta.json"
    all_meta.append(json.loads(mf.read_text()))

(OUTPUT_DIR / "summary.json").write_text(json.dumps(all_meta, indent=2))

# ---- 5) ringkasan ---------------------------------------------------------------
wall = time.time() - t_start
new_pages = sum(j["pages"] for j in jobs)
print("\n" + "=" * 78)
print(f"{'file':34s} {'hal':>4s} {'mode':>8s} {'~detik':>7s} {'eq$$':>5s} {'eq$':>5s} engine")
for m in all_meta:
    print(f"{Path(m['source_pdf']).stem[:33]:34s} {m.get('pages',0):>4} {m.get('mode','-'):>8s} "
          f"{m.get('elapsed_sec_est',0):>7.0f} {m.get('display_equations',0):>5} "
          f"{m.get('inline_equations',0):>5} {m['engine']}")
print("=" * 78)
print(f"Diproses: {len(jobs)} PDF ({new_pages} hal) | cache: {len(skipped)} | "
      f"wall {wall:.0f}s" + (f" | {new_pages / wall:.2f} hal/detik" if new_pages and wall else ""))
bad = [m for m in all_meta if m["engine"] != "marker"]
if bad:
    print("PERINGATAN: file tidak lewat marker:", [Path(m["source_pdf"]).name for m in bad],
          "-> baca log di", LOG_DIR)
print(f"Selesai. Output di {OUTPUT_DIR}")

## Kalibrasi mode (opsional)

In [ ]:
# KALIBRASI (opsional, ~1-2 menit): bandingkan "fast" vs "accurate" pada beberapa halaman pertama
# satu PDF. Gunakan untuk memutuskan KEEP_OCR_FOR_MATH. Waktu sudah termasuk pemuatan model,
# jadi bandingkan terutama jumlah rumus; selisih waktu sebenarnya makin besar pada PDF panjang.
def compare_modes(name_part, page_range="0-5"):
    src = next((p for p in pdfs if name_part.lower() in p.name.lower()), None)
    if src is None:
        raise SystemExit(f"tidak ada PDF yang namanya mengandung '{name_part}'")
    res = {}
    for mode in ("fast", "accurate"):
        base = WORK / "_cal" / mode
        shutil.rmtree(base, ignore_errors=True)
        stg, raw = base / "in", base / "out"
        stg.mkdir(parents=True); raw.mkdir(parents=True)
        shutil.copy2(src, stg / "cal.pdf")
        cmd = build_cmd(mode, stg, raw, 1, extra=("--page_range", page_range))
        t0 = time.time()
        r = subprocess.run(cmd, env=build_env(devices[0]), capture_output=True, text=True)
        dt = time.time() - t0
        mds = list(raw.rglob("*.md"))
        if r.returncode != 0 or not mds:
            print(mode, "gagal:\n", (r.stdout + r.stderr)[-1500:]); return
        res[mode] = (dt, qc_report(mds[0].read_text(encoding="utf-8"), 0))
    print(f"{src.name}  halaman {page_range}")
    print(f"{'mode':9s} {'detik':>6s} {'chars':>7s} {'eq$$':>5s} {'eq$':>5s} {'tabel':>6s}")
    for mode, (dt, q) in res.items():
        print(f"{mode:9s} {dt:>6.0f} {q['chars']:>7} {q['display_equations']:>5} "
              f"{q['inline_equations']:>5} {q['markdown_table_rows']:>6}")
    f, a = res["fast"][1], res["accurate"][1]
    gap = a["inline_equations"] - f["inline_equations"]
    if gap > max(2, 0.2 * a["inline_equations"]):
        print("-> accurate menemukan lebih banyak rumus inline: biarkan KEEP_OCR_FOR_MATH = True.")
    else:
        print("-> fast setara untuk PDF ini: KEEP_OCR_FOR_MATH = False aman dan lebih cepat.")

# compare_modes("jurio")        # ganti dengan potongan nama PDF kamu

## Inspeksi hasil

In [ ]:
import json, pathlib, re
OUT = pathlib.Path(OUTPUT_DIR)
for d in sorted(p for p in OUT.iterdir() if p.is_dir()):
    mfile = d / f"{d.name}_meta.json"
    if not mfile.exists():
        print(f"[skip] {d.name}: tidak ada meta"); continue
    meta = json.loads(mfile.read_text())
    print("=" * 70)
    for k in ["source_pdf", "engine", "mode", "mode_reason", "pages", "elapsed_sec_est", "headings",
              "display_equations", "inline_equations", "markdown_table_rows", "images_saved",
              "references_section_found", "warning", "warning_low_text"]:
        if k in meta:
            print(f"{k:26s}: {meta[k]}")

# Sampel equation / tabel / gambar. Ganti idx untuk file lain.
idx = 0
dirs = sorted(p for p in OUT.iterdir() if p.is_dir())
d = dirs[idx]
print("\ndokumen:", d.name)
md_text = next(d.glob("*.md")).read_text()
eqs = re.findall(r"\$\$(.+?)\$\$", md_text, flags=re.S)
print(f"--- {len(eqs)} display equations, 3 pertama ---")
for e in eqs[:3]:
    print(e.strip()[:200], "\n")
rows = [l for l in md_text.splitlines() if l.startswith("|")]
print(f"--- {len(rows)} baris tabel, 8 pertama ---")
print("\n".join(rows[:8]))
imgs = sorted((d / "images").glob("*")) if (d / "images").exists() else []
print(f"--- {len(imgs)} gambar ---")
for i in imgs[:10]:
    print(i.name)

## Unduh

In [ ]:
# Zip hasil untuk diunduh dari panel Output (Kaggle) / Files (Colab)
import shutil
shutil.make_archive(str(WORK / "pdf2md_output"), "zip", str(OUTPUT_DIR))
print("done:", WORK / "pdf2md_output.zip")